# Federated GraphSAGE for Illicit Transaction Detection

## 1. Dataset Exploration

We use the Elliptic Bitcoin transaction dataset to investigate
graph-based illicit transaction detection.

The dataset contains transaction features, transaction
relationships, and known transaction labels.

In [6]:
import pandas as pd

In [7]:
classes_path = "data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv"

classes = pd.read_csv(classes_path)

print("Shape:", classes.shape)
print("\nColumns:")
print(classes.columns.tolist())

print("\nFirst 5 rows:")
display(classes.head())

Shape: (203769, 2)

Columns:
['txId', 'class']

First 5 rows:


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


In [8]:
print(classes["class"].value_counts())

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


In [9]:
edges_path = "data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"

edges = pd.read_csv(edges_path)

print("Shape:", edges.shape)

print("\nColumns:")
print(edges.columns.tolist())

print("\nFirst 5 rows:")
display(edges.head())

Shape: (234355, 2)

Columns:
['txId1', 'txId2']

First 5 rows:


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


In [12]:
features_sample = pd.read_csv(
    features_path,
    header=None,
    nrows=5
)

print("Sample shape:", features_sample.shape)

print("\nFirst 10 columns:")
display(features_sample.iloc[:, :10])

print("\nLast 5 columns:")
display(features_sample.iloc[:, -5:])

Sample shape: (5, 167)

First 10 columns:


,0,1,2,3,4,5,6,7,8,9
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523



Last 5 columns:


,162,163,164,165,166
0,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,0.277775,0.326394,1.293750,0.178136,0.179117


In [13]:
feature_columns = (
    ["txId", "time_step"]
    + [f"feature_{i}" for i in range(1, 166)]
)

features_sample.columns = feature_columns

print("Number of columns:", len(features_sample.columns))
print("\nFirst 10 columns:")
print(features_sample.columns[:10].tolist())

print("\nLast 5 columns:")
print(features_sample.columns[-5:].tolist())

display(features_sample.head())

Number of columns: 167

First 10 columns:
['txId', 'time_step', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8']

Last 5 columns:
['feature_161', 'feature_162', 'feature_163', 'feature_164', 'feature_165']


,txId,time_step,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [14]:
print("Classes shape:", classes.shape)

print("\nClass columns:")
print(classes.columns.tolist())

print("\nFirst 10 rows:")
display(classes.head(10))

print("\nClass distribution:")
print(classes["class"].value_counts())

Classes shape: (203769, 2)

Class columns:
['txId', 'class']

First 10 rows:


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown
5,230459870,unknown
6,230333930,unknown
7,230595899,unknown
8,232013274,unknown
9,232029206,2



Class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


In [15]:
edges_path = "data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"

edges = pd.read_csv(edges_path)

print("Edges shape:", edges.shape)

print("\nColumn names:")
print(edges.columns.tolist())

print("\nFirst 10 edges:")
display(edges.head(10))

Edges shape: (234355, 2)

Column names:
['txId1', 'txId2']

First 10 edges:


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206
5,232344069,27553029
6,36411953,230405052
7,34194980,5529846
8,3881097,232457116
9,230409257,32877982


In [16]:
print("Number of edges:", len(edges))

print("Unique source transactions:", edges["txId1"].nunique())
print("Unique destination transactions:", edges["txId2"].nunique())

all_nodes = set(edges["txId1"]) | set(edges["txId2"])

print("Unique transactions appearing in edges:", len(all_nodes))

Number of edges: 234355
Unique source transactions: 166345
Unique destination transactions: 148447
Unique transactions appearing in edges: 203769


In [17]:
print("Minimum txId:", min(all_nodes))
print("Maximum txId:", max(all_nodes))

Minimum txId: 1076
Maximum txId: 403244581


In [18]:
# Count how many connections each transaction has

source_counts = edges["txId1"].value_counts()
destination_counts = edges["txId2"].value_counts()

print("Average outgoing connections:",
      source_counts.mean())

print("Average incoming connections:",
      destination_counts.mean())

print("\nMaximum outgoing connections:",
      source_counts.max())

print("Maximum incoming connections:",
      destination_counts.max())

Average outgoing connections: 1.4088490787219334
Average incoming connections: 1.578711594036929

Maximum outgoing connections: 472
Maximum incoming connections: 284


In [19]:
# Find some transactions with many connections

print("Transactions with the most outgoing connections:")
display(source_counts.head(10))

print("\nTransactions with the most incoming connections:")
display(destination_counts.head(10))

Transactions with the most outgoing connections:


txId1
2984918    472
89273      288
102570     122
3181       112
7952        99
1891081     95
143705      92
565334      90
488266      88
793584      82
Name: count, dtype: int64


Transactions with the most incoming connections:


txId2
43388675     284
68705820     247
30699343     241
96576418     239
225859042    212
279187194    211
234890810    199
196107869    188
43397277     182
68706499     178
Name: count, dtype: int64

In [20]:
features_sample = pd.read_csv(
    features_path,
    header=None,
    nrows=1000
)

features_sample.columns = feature_columns

print("Sample shape:", features_sample.shape)

print("\nData types:")
print(features_sample.dtypes.value_counts())

print("\nTime-step range in sample:")
print(
    features_sample["time_step"].min(),
    "to",
    features_sample["time_step"].max()
)

print("\nFeature statistics:")
display(
    features_sample[
        ["feature_1", "feature_2", "feature_3",
         "feature_4", "feature_5"]
    ].describe()
)

Sample shape: (1000, 167)

Data types:
float64    165
int64        2
Name: count, dtype: int64

Time-step range in sample:
1 to 1

Feature statistics:


,feature_1,feature_2,feature_3,feature_4,feature_5
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.039740,-0.151794,-0.728515,0.125654,0.234641
std,0.889864,0.177285,0.782537,1.447287,8.231080
min,-0.172979,-0.210553,-1.756361,-0.121970,-0.063725
25%,-0.172017,-0.184668,-1.201369,-0.121970,-0.043875
50%,-0.167356,-0.184668,-1.201369,-0.121970,-0.043875
75%,-0.151507,-0.182456,-0.091383,-0.046932,-0.043875
max,13.222433,2.456149,1.573595,28.317246,260.090707


In [21]:
print("Feature txId type:", features_sample["txId"].dtype)
print("Class txId type:", classes["txId"].dtype)
print("Edge txId1 type:", edges["txId1"].dtype)
print("Edge txId2 type:", edges["txId2"].dtype)

Feature txId type: int64
Class txId type: int64
Edge txId1 type: int64
Edge txId2 type: int64


In [22]:
feature_ids = set(features_sample["txId"])

class_ids = set(classes["txId"])

print("Feature sample IDs:", len(feature_ids))
print("Class IDs:", len(class_ids))

print(
    "Feature sample IDs found in classes:",
    len(feature_ids & class_ids)
)

Feature sample IDs: 1000
Class IDs: 203769
Feature sample IDs found in classes: 1000
